# Structure and dynamic range

The earlier notebooks of #166 measured a matrix by its condition number, its
spectrum and its factorizations. This one looks at the *magnitudes* of its
entries: where they sit, how they are arranged, and how that meets the
number system the matrix is stored in.

IEEE formats have the same relative precision across their whole normal range;
past it they lose digits in the subnormals, underflow to 0 or overflow to ∞.
Posits and takums have tapered precision: most accurate near 1 and fewer
digits further away, and instead of overflowing or underflowing they saturate
at their largest and smallest values. The two make the trade differently: at
16 bits, posits keep more digits over a middle band of magnitudes, and takums
keep a couple of digits over a far wider range. So which format holds a matrix
best depends on where its entries sit.

This notebook:

1. **Measures the precision of each format against magnitude**, the curve the
   rest of the notebook reads against.
2. **Draws magnitude heatmaps** of log₁₀|A|, log₁₀|A⁻¹| and the LU factors, for
   four matrices whose structure shows: grading, decay, growth and smoothness.
3. **Places each matrix's entries and eigenvalues on those curves**, as
   histograms on the same magnitude axis.
4. **Rounds each matrix and its inverse to each format** and counts the digits
   kept.
5. **Scales one matrix by powers of 2**, which leaves an IEEE format's relative
   error unchanged until it overflows or underflows, and moves a posit's.

The formats are those of #166's decision 4, but for `dd_cascade`. Its values
cross to Python rounded to float64, so its rounding errors are below what this
notebook can measure. Every random matrix has a seed, so the figures are the
same on every build.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scipy.sparse as sp

import mtl5
import mtl5.sparse as ms

g = mtl5.generators

# Chart styling, shared with the other notebooks.
SURFACE = "#fcfcfb"
INK, INK_2, RULE = "#0b0b0b", "#52514e", "#a9a8a3"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]

plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": "#d8d7d2",
        "axes.labelcolor": INK_2,
        "axes.titlecolor": INK,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.color": "#e8e7e2",
        "grid.linewidth": 0.8,
        "text.color": INK,
        "xtick.color": INK_2,
        "ytick.color": INK_2,
        "font.size": 10,
        "figure.dpi": 120,
    }
)

FAMILY = {"IEEE": SERIES[0], "posit": SERIES[1], "takum": SERIES[2]}
WIDTHS = {
    "16 bits": [("fp16", "IEEE"), ("posit16", "posit"), ("takum16", "takum")],
    "32 bits": [("f32", "IEEE"), ("posit32", "posit"), ("takum32", "takum")],
    "64 bits": [("f64", "IEEE")],
}
FORMATS = [name for group in WIDTHS.values() for name, _ in group]
ROUNDED = [name for name in FORMATS if name != "f64"]  # f64 holds the reference


def to_format(x, dtype):
    # Round float64 values to `dtype` and bring them back as float64. f32's
    # to_numpy() returns float32, so promote before any arithmetic.
    x = np.asarray(x, dtype=np.float64)
    y = mtl5.convert(x.ravel(), dtype).to_numpy()
    return np.asarray(y, dtype=np.float64).reshape(x.shape)


def rel_error(x, dtype):
    # Relative rounding error of each entry: inf where the format overflows to
    # inf or returns NaN, 1 where it flushes a nonzero to 0.
    x = np.asarray(x, dtype=np.float64)
    y = to_format(x, dtype)
    with np.errstate(divide="ignore", invalid="ignore"):
        err = np.abs(y - x) / np.abs(x)
    return np.where(np.isfinite(y), err, np.inf)


def digits(err):
    # Correct decimal digits: 17 for an exact entry, 0 for none (including an
    # overflow, a flush to zero and a saturation far from the value).
    with np.errstate(divide="ignore"):
        return np.clip(-np.log10(err), 0, 17) + 0.0

## 1. Precision against magnitude

For each magnitude x on a grid from 10⁻⁵⁰ to 10⁵⁰, 64 seeded random values in
[x, 2x) are rounded to the format, and the largest relative rounding error is
kept. Its −log₁₀ is the number of correct decimal digits the format guarantees
there. The samples are float64, so f64's own rounding error can't be measured
this way: its curve is half the float64 spacing, `np.spacing(x) / 2`, relative
to x. A value the format can't hold has no correct digits: overflow to ∞, a
flush to 0, and a posit or takum's saturation at its largest or smallest value
all show as zero digits.

In [ ]:
rng = np.random.default_rng(0)
mags = np.logspace(-50, 50, 401)
samples = mags[:, None] * (1 + rng.random((1, 64)))

precision = {}
for name in FORMATS:
    if name == "f64":
        err = np.spacing(samples) / 2 / samples
    else:
        err = rel_error(samples, name)
    precision[name] = digits(err.max(axis=1))

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), sharey=True)
for ax, (title, group), where in zip(axes, WIDTHS.items(), ["upper right"] * 2 + ["lower right"]):
    for name, family in group:
        ax.semilogx(mags, precision[name], color=FAMILY[family], linewidth=2, label=name)
    ax.axvline(1, color=RULE, linestyle=":", linewidth=1)
    ax.set_ylim(-0.5, 17)
    ax.set_xlabel("magnitude |x|")
    ax.set_title(title, loc="left", fontsize=11)
    ax.legend(frameon=False, fontsize=8, loc=where)
axes[0].set_ylabel("correct decimal digits")
fig.tight_layout()
plt.show()


LOGS = np.log10(mags)


def digits_at(name, x):
    return precision[name][np.argmin(np.abs(LOGS - np.log10(x)))]


def band_around_1(a, b):
    # The contiguous band of magnitudes around 1 where format a keeps at least
    # as many digits as format b.
    ok = precision[a] >= precision[b]
    lo = hi = int(np.argmin(np.abs(LOGS)))
    while lo > 0 and ok[lo - 1]:
        lo -= 1
    while hi < LOGS.size - 1 and ok[hi + 1]:
        hi += 1
    return mags[lo], mags[hi]


print("digits at   1e-50  1e-20   1e-8      1    1e8   1e20   1e50")
for name in FORMATS:
    row = "  ".join(f"{digits_at(name, x):5.1f}" for x in (1e-50, 1e-20, 1e-8, 1, 1e8, 1e20, 1e50))
    print(f"  {name:8s}", row)
print()
for a, b in [
    ("posit16", "fp16"),
    ("takum16", "fp16"),
    ("posit16", "takum16"),
    ("posit32", "f32"),
    ("takum32", "f32"),
]:
    lo, hi = band_around_1(a, b)
    print(f"{a} keeps at least {b}'s digits for |x| in [{lo:.1e}, {hi:.1e}]")

The curves read as follows:

- **IEEE** formats are flat: fp16 keeps about 3.4 digits, f32 about 7.3 and
  f64 about 16, wherever a value is normal. fp16's range is narrow: it loses
  digits in its subnormals below 6.1·10⁻⁵ (2⁻¹⁴), rounds values at or below
  3·10⁻⁸ (2⁻²⁵, half its smallest subnormal) to zero, and overflows to ∞
  from 65520 on (its largest finite value is 65504).
- **posit16 and takum16** keep 3.7 digits near 1, a little more than fp16.
  posit16 keeps at least fp16's digits from about 0.006 to 100, and takum16
  only from about 0.2 to 3: between there and the ends of fp16's range, fp16
  is more precise than takum16. Beyond them, posit16 tapers to nothing by its
  saturation points, about 10⁻¹⁷ and 7·10¹⁶, while takum16 keeps about two
  digits out to 10⁻⁵⁰ and 10⁵⁰ and beyond.
- **posit32 and takum32** keep 8.5 digits near 1, more than f32. posit32
  keeps at least f32's digits from about 10⁻⁶ to 3·10⁵ and takum32 from
  about 10⁻⁹ to 10⁹. Outside those bands f32 is more precise until it runs
  out of range near 10⁻³⁸ (subnormals) and 3.4·10³⁸. posit32 saturates
  near 10⁻³⁶ and 10³⁶; takum32 keeps more than six digits out to 10⁻⁵⁰ and 10⁵⁰.

So the tapered formats win near 1 and lose further out, until the IEEE format
runs out of range. What that means for a matrix depends on where its entries
are.

## 2. Magnitude heatmaps

Four matrices of order 8 to 16, each chosen for the structure its magnitudes
show:

- **`hilbert(8)`**: smooth entries 1/(i + j − 1) between 0.07 and 1, and an
  inverse with entries up to 10⁹ in a checkerboard of signs;
- **`kahan(16, 0.5)`**: upper triangular, with rows graded by powers of
  sin θ down to 1.6·10⁻⁵;
- **`pascal(10)`**: binomial coefficients growing to 4.9·10⁴ away from the
  first row and column;
- **a diagonally dominant tridiagonal matrix**, tridiag(−1, 4, −1) of order 16,
  whose inverse decays exponentially away from the diagonal.

Their condition numbers are at most about 4·10⁹, so their inverses, computed
in f64 by `mtl5.inv`, are accurate to several digits, more than these log
scales show. Each panel has its own colour scale, so the structure inside
each matrix shows; the printed ranges compare them. White cells are exact
zeros.

In [ ]:
n_tri = 16
tridiag = 4 * np.eye(n_tri) - np.eye(n_tri, k=1) - np.eye(n_tri, k=-1)
MATRICES = {
    "hilbert(8)": g.hilbert(8).to_numpy(),
    "kahan(16, 0.5)": g.kahan(16, 0.5).to_numpy(),
    "pascal(10)": g.pascal(10).to_numpy(),
    "tridiag(−1, 4, −1)": tridiag,
}
INVERSES = {name: mtl5.inv(mtl5.matrix(A)).to_numpy() for name, A in MATRICES.items()}

cmap = plt.get_cmap("Blues").copy()
cmap.set_bad(SURFACE)


def log_magnitude(M):
    with np.errstate(divide="ignore"):
        return np.ma.masked_invalid(np.where(M == 0, np.nan, np.log10(np.abs(M))))


def heatmap(ax, M, title):
    im = ax.matshow(log_magnitude(M), cmap=cmap)
    ax.set_title(title, loc="left", fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
    ax.figure.colorbar(im, ax=ax, shrink=0.8, label="log₁₀ magnitude")


fig, axes = plt.subplots(len(MATRICES), 2, figsize=(8.5, 14), layout="constrained")
for row, (name, A) in enumerate(MATRICES.items()):
    heatmap(axes[row, 0], A, f"{name}: log₁₀ |A|")
    heatmap(axes[row, 1], INVERSES[name], f"{name}: log₁₀ |A⁻¹|")
plt.show()

for name, A in MATRICES.items():
    Ai = INVERSES[name]
    nz, nzi = np.abs(A[A != 0]), np.abs(Ai[Ai != 0])
    print(
        f"{name:19s} kappa = {np.linalg.cond(A):.1e}  "
        f"|A| in [{nz.min():.1e}, {nz.max():.1e}]  |A^-1| in [{nzi.min():.1e}, {nzi.max():.1e}]"
    )

The structure is visible at once:

- Hilbert's matrix is smooth and near 1, and its inverse has entries from
  64 to 4.2·10⁹, largest toward the lower right, away from the first row and
  column.
- Kahan's rows fade geometrically, and its inverse grows toward the top right,
  to 3.7·10⁸: solving with a graded triangular matrix amplifies the small rows.
- Pascal's entries grow toward the lower right corner. Its inverse, an
  integer matrix too, is largest in the middle and smallest at the corners.
- The tridiagonal matrix's inverse decays by about the same factor per step
  away from the diagonal, the exponential decay that diagonal dominance
  guarantees for banded matrices (Demko, Moss and Smith, 1984).

Partial pivoting rearranges these magnitudes once more. Below are the L and U
of Hilbert's and Pascal's matrices, from `mtl5.sparse.splu` with natural
column order and a pivot threshold of 1, which is classic partial pivoting.

In [ ]:
def lu_factors(A):
    F = ms.splu(ms.from_scipy(sp.csr_matrix(A)), ordering="natural", threshold=1.0)
    return ms.to_scipy(F.L).toarray(), ms.to_scipy(F.U).toarray()


fig, axes = plt.subplots(2, 2, figsize=(8.5, 7), layout="constrained")
for row, name in enumerate(["hilbert(8)", "pascal(10)"]):
    L, U = lu_factors(MATRICES[name])
    heatmap(axes[row, 0], L, f"{name}: log₁₀ |L|")
    heatmap(axes[row, 1], U, f"{name}: log₁₀ |U|")
    d = np.abs(np.diag(U))
    print(
        f"{name:11s} |L| <= {np.abs(L).max():.2g}, |u_kk| from {d.max():.1e} down to {d.min():.1e}"
    )
plt.show()

L's entries are at most 1 in magnitude, which is what partial pivoting
guarantees. U carries the magnitudes instead, and its diagonal falls with k:
Hilbert's pivots fall over nine orders of magnitude, to 1.2·10⁻⁹, and Pascal's
over three, as the stability notebook's pivot profiles showed for other
matrices. A low-precision format that holds A well can still
lose the trailing pivots of U.

## 3. Where the magnitudes sit

The histograms below put each matrix's entries, its inverse's entries, and its
eigenvalues on the magnitude axis of section 1, with the 16-bit precision
curves above them. The 16-bit formats are where the differences are largest.

The eigenvalues are from `mtl5.eigvalsh` for the three symmetric matrices; for
Kahan's triangular matrix they are its diagonal. With κ at most 4·10⁹ and the
smallest eigenvalue at least 10⁻¹¹, `eigvalsh`'s absolute error, a small
multiple of u‖A‖, leaves each eigenvalue's magnitude accurate to the bin.

In [ ]:
EIGS = {}
for name, A in MATRICES.items():
    if name.startswith("kahan"):
        EIGS[name] = np.diag(A)
    else:
        EIGS[name] = np.asarray(mtl5.eigvalsh(mtl5.matrix(A)), dtype=np.float64)

bins = np.logspace(-12, 10, 45)
fig, axes = plt.subplots(
    1 + len(MATRICES),
    1,
    figsize=(8, 11),
    sharex=True,
    gridspec_kw={"height_ratios": [1.6] + [1] * len(MATRICES)},
)
for name, family in WIDTHS["16 bits"]:
    axes[0].semilogx(mags, precision[name], color=FAMILY[family], linewidth=2, label=name)
axes[0].set_ylim(-0.2, 4.5)
axes[0].set_ylabel("digits")
axes[0].set_title(
    "16-bit precision, and where four matrices' magnitudes sit", loc="left", fontsize=11
)
axes[0].legend(frameon=False, fontsize=8, loc="upper right")
kinds = [("entries of A", SERIES[0]), ("entries of A⁻¹", SERIES[1]), ("|eigenvalues|", INK_2)]
for ax, name in zip(axes[1:], MATRICES):
    A, Ai, lam = MATRICES[name], INVERSES[name], EIGS[name]
    for (label, color), vals in zip(kinds, [A[A != 0], Ai[Ai != 0], lam]):
        ax.hist(
            np.abs(vals),
            bins=bins,
            color=color,
            alpha=0.55 if color != INK_2 else 0.9,
            histtype="stepfilled" if color != INK_2 else "step",
            linewidth=1.5,
            label=label,
        )
    ax.set_ylabel("count")
    ax.set_title(name, loc="left", fontsize=10)
axes[1].legend(frameon=False, fontsize=8, loc="upper right")
axes[-1].set_xlim(bins[0], bins[-1])
axes[-1].set_xlabel("magnitude")
fig.tight_layout()
plt.show()

Each matrix puts something outside the band where a 16-bit format is at its
best:

- **Hilbert's** entries sit between 0.07 and 1, but its inverse reaches
  4.2·10⁹, past fp16's overflow, and its smallest eigenvalue is about 10⁻¹⁰.
- **Kahan's** entries span five decades below 1, down into fp16's
  subnormals, and its inverse reaches 3.7·10⁸.
- **Pascal's** entries and its inverse's lie between 1 and 5·10⁴, inside
  fp16's range, and many of them above 100, where posit16 and takum16 have
  fallen below fp16.
- **The tridiagonal** matrix holds small integers, but its inverse's decaying
  tail reaches 6.6·10⁻¹⁰, far below the 3·10⁻⁸ at which fp16 rounds to zero.

## 4. Rounding each matrix to each format

Each matrix and its inverse is rounded to each format, and the median and
the worst number of correct digits over the nonzero entries are recorded. An
entry that overflows or flushes to zero has no correct digits. f64 is left
out: the matrices are generated in float64, which is the reference.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True, layout="constrained")
table = {}
for ax, (title, source) in zip(axes, [("A", MATRICES), ("A⁻¹", INVERSES)]):
    med = np.zeros((len(source), len(ROUNDED)))
    worst = np.zeros_like(med)
    for i, (name, M) in enumerate(source.items()):
        vals = M[M != 0]
        for j, fmt in enumerate(ROUNDED):
            d = digits(rel_error(vals, fmt))
            med[i, j], worst[i, j] = np.median(d), d.min()
    table[title] = (med, worst)
    im = ax.imshow(med, cmap="Blues", vmin=0, vmax=9, aspect="auto")
    for i in range(med.shape[0]):
        for j in range(med.shape[1]):
            ax.text(
                j,
                i,
                f"{med[i, j]:.1f}\n({worst[i, j]:.1f})",
                ha="center",
                va="center",
                fontsize=7,
                color=SURFACE if med[i, j] > 5 else INK,
            )
    ax.set_xticks(range(len(ROUNDED)), ROUNDED, rotation=30)
    ax.set_yticks(range(len(source)), list(source))
    ax.grid(False)
    ax.set_title(f"digits kept in the entries of {title}: median (worst)", loc="left", fontsize=10)
fig.colorbar(im, ax=axes, shrink=0.85, label="median correct digits")
plt.show()

for title, (med, worst) in table.items():
    print(f"worst-case digits in {title}:")
    for i, name in enumerate(MATRICES):
        print(
            f"  {name:19s}", "  ".join(f"{fmt} {worst[i, j]:5.1f}" for j, fmt in enumerate(ROUNDED))
        )

Reading across the 16-bit columns:

- For the **entries of A**, posit16 keeps the most digits on Hilbert's matrix,
  whose entries sit near 1. On Pascal's, whose large binomials are not exact
  in any 16-bit format, fp16's worst entry keeps 3.5 digits and the tapered
  formats' 2.9. Kahan's smallest entries are subnormal in fp16, and every
  16-bit format's worst entry keeps under three digits. The tridiagonal
  matrix's integers are exact everywhere.
- For the **inverses**, fp16 overflows on Hilbert's and Kahan's largest
  entries and flushes the tridiagonal inverse's tail to zero: those entries
  keep no correct digits. posit16 and takum16 hold every entry, takum16 with
  at least 2.5 digits and posit16 with as few as 1.6.
- **At 32 bits**, f32 holds every entry with its full precision. posit32 and
  takum32 keep at least f32's median digits everywhere, but posit32's worst
  entry keeps fewer than f32's in three of the four inverses, as few as 6.4
  digits for Hilbert's.

So the worst column of the table, not the median, decides whether a format
can store a matrix at all; the median decides how well.

## 5. Scaling moves a posit's accuracy, not an IEEE format's

Multiplying a matrix by a power of 2 is exact in binary floating point, and an
IEEE format's relative rounding error doesn't change until the scaled entries
overflow or underflow. For a tapered format it does: the entries move along
the precision curve of section 1.

The test matrix is `randorth(32, seed=3)`, an orthogonal matrix whose entries
are about 0.13 in median, scaled by 2ᵏ for k from −60 to 60. The curve is the
median number of digits kept over its entries.

In [ ]:
Q = g.randorth(32, seed=3).to_numpy()
ks = np.arange(-60, 61, 2)
scaled = {name: [] for name in FORMATS if name != "f64"}
for k in ks:
    vals = np.ldexp(Q[Q != 0], int(k))
    for name in scaled:
        scaled[name].append(np.median(digits(rel_error(vals, name))))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, title in zip(axes, ["16 bits", "32 bits"]):
    for name, family in WIDTHS[title]:
        ax.plot(
            ks,
            scaled[name],
            color=FAMILY[family],
            linewidth=2,
            marker="o",
            markersize=3,
            label=name,
        )
    ax.axvline(0, color=RULE, linestyle=":", linewidth=1)
    ax.set_xlabel("k, the scaling 2ᵏ")
    ax.set_title(f"{title}: digits kept in 2ᵏ Q", loc="left", fontsize=11)
    ax.legend(frameon=False, fontsize=8, loc="lower center")
axes[0].set_ylabel("median correct digits")
fig.tight_layout()
plt.show()

for name, vals in scaled.items():
    best = ks[int(np.argmax(vals))]
    print(
        f"{name:8s} at k = 0: {vals[list(ks).index(0)]:.1f} digits; best {max(vals):.1f} "
        f"at k = {best}; at k = -40: {vals[list(ks).index(-40)]:.1f}; "
        f"at k = 40: {vals[list(ks).index(40)]:.1f}"
    )

fp16 and f32 are flat until the scaled entries leave their range: fp16
between about k = −10 and 18, and f32 across the whole plot. posit16 and
takum16 peak at k = 4, where the median entry, 0.13 · 2⁴ ≈ 2, is near 1, and
fall away on both sides. Inside fp16's range, posit16 beats fp16 only from
k = −2 to 8, and takum16 only at k = 2 and 4; once fp16 is out of range, both
keep some digits and takum16 the most. At 32 bits, posit32 and takum32 beat
f32 from k = −14 to about 22 and 24, and f32 wins beyond.

For a posit or takum, then, scaling is not free: a diagonal scaling that
brings a matrix's entries toward 1 buys precision, and one that pushes them
away costs it. For an IEEE format it only matters at the edges of the range.

## What to take away

- **Precision is a function of magnitude.** IEEE formats are flat inside their
  range and fail at its edges; posits and takums are best near 1 and taper
  away from it, posits faster than takums.
- **The structure of a matrix decides where its magnitudes are.** Graded rows,
  growing binomials, decaying inverses and falling pivots each place entries
  far from 1, in A, in A⁻¹ or in U, even when A itself looks harmless.
- **Read the worst entry, not only the median.** fp16 kept a median of over
  three digits in Hilbert's matrix and none in the largest entries of its
  inverse.
- **For tapered formats, scaling is part of the precision.** Equilibrating a
  matrix toward 1 gains posit and takum digits; it changes nothing for IEEE
  formats inside their range.

### Where to go next

- `conditioning.ipynb`: the condition number each format can hold.
- `stability.ipynb`: pivot profiles and the growth factor.
- Coming in #166: componentwise (Skeel) conditioning under row and column
  scaling, and the `mtl5.plot` module.